# Testing `fairscape_models.sql`

In [1]:
import pathlib
import sys
import os

In [2]:
try:
	os.remove("integration_test.db")
except:
	pass

In [3]:
# load in the fairscape models library
sys.path.insert(0, '/workspaces/fairscape_models')

In [4]:
from fairscape_models.utils import readCrate

from fairscape_models.sql.models import *
from fairscape_models.sql.ingest import ROCrateIngestRequest

In [5]:
import sqlalchemy as sa

## Load ROCrate Tests

In [6]:
releases = [ elem for elem in pathlib.Path("/mnt/data/Dataverse/").glob("*") if elem.is_dir()]
test_release = pathlib.Path("/mnt/data/Dataverse/U2OS")

In [7]:
test_rocrate_list = list(test_release.glob("*.zip"))
test_rocrate_path = test_release / "cm4ai_u2os_1_ImageDownloader.zip"


In [8]:
test_rocrate = readCrate(test_rocrate_path)

In [9]:
len(test_rocrate.metadataGraph)

20552

In [10]:
# TODO create a flamegraph of loading rocrate
# py-spy
# pip install py-spy
# py-spy record -o profile.svg -- python myscript.py 
#
# flameprof for cProfile stats
# python -m cProfile -o script.prof myscript.py
# 

In [11]:
# create an engine
engine = sa.create_engine("sqlite:///integration_test.db")


# create table 
Base.metadata.create_all(engine)



In [12]:
# create a session
session = sa.orm.Session(engine) 

ingestRequest = ROCrateIngestRequest(
	model=test_rocrate,
	session=session,
)

In [13]:
# test digest authors
author_data = ingestRequest._digest_authors()



In [14]:
# check authors
author_data, existing_author_ids = ingestRequest._check_authors(author_data)

# if no id's are found author id is an empty dictionary
existing_author_ids

{}

In [15]:
# write the authors
author_ids = ingestRequest._write_authors(author_data)

# join existing author ids to author_ids
full_author_ids = author_ids | existing_author_ids

In [16]:
full_author_ids

{'Yue Qin': 1,
 'Abantika Pal': 2,
 'Trang Le': 3,
 'Dorothy Tsai': 4,
 'Jing Chen': 5,
 'Joanna Lenkiewicz': 6,
 'J. Wade Harper': 7,
 'Laura Pontano Vaites': 8,
 'Aji Palar': 9,
 'William Leineweber': 10,
 'Anthony Cesnik': 11,
 'Mengzhou Hu': 12,
 'Andrew P. Latham': 13,
 'Kyung-Mee Moon': 14,
 'Ishan Gaur': 15,
 'Andrej Sali': 16,
 'Keiichiro Ono': 17,
 'Leah V. Schaffer': 18,
 'Ignacia Echeverria': 19,
 'Leonard J. Foster': 20,
 'Steven P. Gygi': 21,
 'Emma Lundberg ': 22,
 'Christopher Churas': 23,
 'Peter Zage': 24,
 'Neelesh Soni': 25,
 'Gege Qian': 26,
 'Trey Ideker': 27,
 'Nicole M. Mattson': 28,
 'Katherine Licon': 29,
 'Robin Bachelder': 30,
 'Edward L. Huttlin': 31,
 'Ernst Pulido': 32,
 'Dexter Pratt': 33,
 'Xiaoyu Zhao': 34}

In [17]:
author_ids

{'Yue Qin': 1,
 'Abantika Pal': 2,
 'Trang Le': 3,
 'Dorothy Tsai': 4,
 'Jing Chen': 5,
 'Joanna Lenkiewicz': 6,
 'J. Wade Harper': 7,
 'Laura Pontano Vaites': 8,
 'Aji Palar': 9,
 'William Leineweber': 10,
 'Anthony Cesnik': 11,
 'Mengzhou Hu': 12,
 'Andrew P. Latham': 13,
 'Kyung-Mee Moon': 14,
 'Ishan Gaur': 15,
 'Andrej Sali': 16,
 'Keiichiro Ono': 17,
 'Leah V. Schaffer': 18,
 'Ignacia Echeverria': 19,
 'Leonard J. Foster': 20,
 'Steven P. Gygi': 21,
 'Emma Lundberg ': 22,
 'Christopher Churas': 23,
 'Peter Zage': 24,
 'Neelesh Soni': 25,
 'Gege Qian': 26,
 'Trey Ideker': 27,
 'Nicole M. Mattson': 28,
 'Katherine Licon': 29,
 'Robin Bachelder': 30,
 'Edward L. Huttlin': 31,
 'Ernst Pulido': 32,
 'Dexter Pratt': 33,
 'Xiaoyu Zhao': 34}

In [19]:
# TODO slow?
ingestRequest._write_identifier_authors(full_author_ids)

In [20]:
# check that author table has correct author information
session.scalar(sa.select(sa.func.count(AuthorSQL.id)))

34

In [21]:
# check linked authors
session.scalar(sa.select(sa.func.count(AuthorIdentifierSQL.id)))

698633

In [22]:
# digest identifiers
rocrate_identifiers = ingestRequest._digest_identifiers()
ingestRequest._write_identifiers(rocrate_identifiers)

session.flush()
session.commit()

In [23]:
# check the identifiers

In [24]:
# digest all crate elements 
crate_elements = ingestRequest._digest_iterate_elements()

In [25]:
# write rocrate elements
ingestRequest._write_elements()

In [28]:
session.commit()

In [29]:
session.close()

## Test Query